# Eukaryotic toehold switch — ON/OFF report (single switch + AND)

Built from `and_eu_report_spec.md` in this folder. Measures each candidate's real,
Boltzmann-sampled ON/OFF behaviour through `FoldEngine` only (no NUPACK, no MFE-only
shortcuts) and decomposes every sampled state into three independent signals —
**`trigger_bound`** (paired to an external strand), **`stem_closed`** (the intended OFF
hairpin actually closed), **`misfolded`** (paired to neither) — rather than collapsing
them into one "open" bit. That collapse is exactly what let an earlier version of this
project's AND construct read 80-90% "open" with **zero triggers present**, purely from
intramolecular cross-hairpin misfolding (see Traps, bottom of the spec).

Reuses `fuse`/`measure`/`load_ranked_designs`/`motif_self_bind_score` from
`_switch_construct.py` (migrated out of `toehold_and_eu_test.ipynb` in an earlier commit
on this branch) and the parallel workers + `stem_closed_fraction`/`misfolded_fraction`/
`_frac_paired_to_external`/`pick_fake_design` from `_joint_state_parallel.py` — nothing
here reimplements either module.

**Reading `OPEN_THRESHOLD = 0.70`:** numbers in this notebook are *not* directly
comparable to `toehold_and_eu_test.ipynb`'s reports, which used `0.80`.


## Setup

In [1]:
# Put the shared _fixtures.py (notebooks/ root) and this folder itself
# (_switch_construct.py / _joint_state_parallel.py) on sys.path, then fx.bootstrap()
# adds <repo>/src so `import engine...` resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_fixtures.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()


engine package root: /Users/zeev/Projects/cernal_software/src


PosixPath('/Users/zeev/Projects/cernal_software/src')

## Parameters — every tunable in one place (and_eu_report_spec.md Step 1)

In [2]:
# ============================================================================
# PARAMETERS. If a later cell needs a new threshold, it belongs HERE, not inline.
# ============================================================================

# ---- data source -----------------------------------------------------------
CSV_PATH = (
    "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/"
    "output_T2/targets/sele/basic_switch/results_ranked.csv"
)
# NucSyn runner export, topology euk_open_5p_kozak_after_stem, already sorted
# best-first by postgen_global_rank. Printed on both reports.

# Column-name map load_ranked_designs() needs -- a REQUIRED parameter there, not a
# module default (_switch_construct.py's own docstring: a second copy of a column
# map living inside the shared module is the same drift problem CLAUDE.md sec 1 warns
# about, one level down).
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}
# CSV columns carried through into design["metrics"] unchanged -- never recomputed
# here (CLAUDE.md sec 5: a value already computed upstream is read, not re-derived).
CARRY_METRICS = ("sim_score", "mcc", "switch_toehold_unpaired_prob_mean", "energy_trigger_mfe")

# fuse()'s own required parameters (and_eu_report_spec.md Step 2's AND topology): the
# 5' half keeps everything before its OWN kozak starts, the 3' half keeps everything
# after its OWN prefix ends.
CUT_5P_BEFORE = "kozak"
KEEP_3P_AFTER = "prefix"

# ---- folding -----------------------------------------------------------------
TEMPERATURE_C = 37.0  # folding temperature (deg C), recorded on every energy below

# ---- single-switch screen -----------------------------------------------------
SINGLE_POOL_N = 300    # ranked designs loaded for the single-switch screen
SINGLE_REPORT_N = 20   # how many reach the Single Switch report

# ---- AND grid -------------------------------------------------------------------
AND_GRID_N = 20   # top-N singles (by single-switch ON/OFF ratio) fed into the N x N grid
AND_REPORT_N = 20 # how many AND pairs reach the AND report

# ---- sampling --------------------------------------------------------------------
N_SAMPLES = 1200        # Boltzmann samples drawn per state
OPEN_THRESHOLD = 0.70   # fraction of footprint bound externally to call a state "open"
# 0.70, not the predecessor notebook's 0.80 -- numbers here are NOT directly
# comparable to that notebook's reports (and_eu_report_spec.md Step 5).

# ---- fake-trigger counts -- two separate parameters, same default, different reason -
SINGLE_N_FAKES = 10
# Distinct fake triggers checked per single-switch candidate, averaged into its OFF
# term -- "10 different fake triggers... the mean will be the result". A single switch
# has ONE footprint, so there is only one pool to draw this many fakes from.
AND_N_FAKES = 10
# Distinct fakes drawn PER SIDE for each AND pair -- 10 for the A-side states, 10 for
# the B-side states (20 fake-involving draws per pair, two INDEPENDENT rotations).
# A separate parameter from SINGLE_N_FAKES not because the count differs (both default
# to 10) but because AND has two footprints screened against, drawn from, and reported
# against separately -- changing one side's count must never silently change the
# other's.

# ---- fake-trigger selection (Trap 2 / Trap 3 fixes) -------------------------------
FAKE_MAX_SEQ_OVERLAP_NT = 8  # max complementary run (wobbles included) a fake may have
WOBBLE_MAX_FRACTION = 0.50   # max share of a matched run that may be G.U wobble

# ---- AND spacer --------------------------------------------------------------------
SPACER_VARIANTS = ("no_spacer", "spacer")  # both are run and reported separately
SPACER_MIN_LENGTH = 15
SPACER_MAX_LENGTH = 20
SPACER_SEED = 1                   # literal seed -- never wall-clock (CLAUDE.md sec 6)
SPACER_CANDIDATE_POOL_SIZE = 150  # pool generate_spacer_candidates() draws from

# ---- optional homopolymer filter ----------------------------------------------------
FILTER_HOMOPOLYMER_RUN = False   # exclude toeholds with a run > MAX_HOMOPOLYMER_RUN
MAX_HOMOPOLYMER_RUN = 5

# ---- execution ------------------------------------------------------------------------
MAX_WORKERS = 6   # process-pool width -- one pool call per BATCH, never one per job

# ---- snapshot --------------------------------------------------------------------------
SNAPSHOT_TO_LOAD = None
# Set to a snapshot filename (or an absolute path) to skip every heavy cell below and
# restore single_report_rows/and_report_rows from a previous run instead. None (the
# default) means a normal full run -- every heavy cell actually runs. A `None` default
# here (not a filename) is deliberate: "Run All" must never silently replace a fresh
# computation with stale saved data.


In [3]:
# One FoldEngine for the WHOLE notebook (single-switch part and AND part both reuse
# it) -- one cache, one temperature (CLAUDE.md sec 1/sec 5: FoldEngine is the only
# place a gate family folds, and exactly one instance exists per run).
folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)

# Publish that one engine to the parallel-worker module, which folds through it
# instead of importing ViennaRNA itself. The pools fork, so every worker inherits
# THIS instance -- must happen before any pool starts.
from _joint_state_parallel import use_fold_engine
use_fold_engine(folder)
print(f"FoldEngine published to the worker module: {folder.versions()}")

from _switch_construct import fuse, load_ranked_designs, measure
from _joint_state_parallel import (
    AndDecompJob,
    SingleDecompJob,
    generate_spacer_candidates,
    pick_fake_design,
    pick_optimized_spacer,
    run_parallel_and_decomp,
    run_parallel_single_decomp,
)
from engine.sequences import longest_homopolymer

import datetime, html, json, time
from pathlib import Path


FoldEngine published to the worker module: {'ViennaRNA': '2.7.2', 'temperature_c': '37.0', 'sampling_uniq_ml': '1'}


## Plotting + report CSS helpers

In [4]:
# ==================================================================================
# Plotting helpers -- same conventions as toehold_and_eu_test.ipynb (2D structure via
# FoldEngine.layout_coordinates, a real base-pair-probability heatmap, a segmented
# layout bar for the domain diagram). Nothing here re-folds anything: every sequence
# plotted was already folded by measure()/the decomposition jobs earlier in this run,
# so folder.mfe/.base_pair_probabilities calls below are cache hits.
# ==================================================================================
import base64
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np

_COLOR_A = "#4C6E8C"   # hairpin A / 5' half / single switch's one hairpin
_COLOR_B = "#B5507E"   # hairpin B / 3' half (AND only)
_KOZAK_COLOR = "#2E9E56"

_DOMAIN_COLOR_A = {
    "prefix": "#DCE4D8", "toehold": "#4C6E8C", "stem_up": "#7E97AC",
    "loop": "#C7D0CA", "stem_down": "#7E97AC", "kozak": "#2E9E56", "exp_gene": "#B5760E",
}
# AND needs a doubled palette so hairpin 1 and 2 are visually distinguishable
# (and_eu_report_spec.md Step 4) -- same domain names, a second, B-tinted hue each.
_DOMAIN_COLOR_B = {
    "prefix": "#E3D8E0", "toehold": "#B5507E", "stem_up": "#C98CA8",
    "loop": "#DFC7D4", "stem_down": "#C98CA8", "kozak": "#2E9E56", "exp_gene": "#8E6A3A",
}


def _png_from_fig(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", facecolor="white")
    plt.close(fig)
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("ascii")


def structure_png(sequence, structure, title, spans=(), figsize=(3.6, 3.2)):
    """One folded structure as a base64 PNG, with `spans` ((start,end,color) tuples)
    outlined. Coordinates from FoldEngine.layout_coordinates (the shared 2D layout)."""
    points = folder.layout_coordinates(structure)
    xs = [x for x, _ in points]
    ys = [y for _, y in points]

    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    ax.plot(xs, ys, "-", color="#B9C2BC", linewidth=1.1, zorder=1)
    stack = []
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            ax.plot([xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                     linewidth=0.9, zorder=1, alpha=0.7)
    for i in range(len(sequence)):
        ring = next((c for s, e, c in spans if s <= i < e), None)
        ax.scatter(xs[i], ys[i], s=24 if ring else 13, color="#C7D0CA",
                    edgecolors=ring or "#666666", linewidths=1.3 if ring else 0.3,
                    zorder=3 if ring else 2)
    ax.text(xs[0], ys[0] + 8, "5'", fontsize=7, ha="center", color="#345")
    ax.text(xs[-1], ys[-1] + 8, "3'", fontsize=7, ha="center", color="#345")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    return _png_from_fig(fig)


def bpp_heatmap_png(sequence, matrix, title, spans=(), figsize=(3.4, 3.1)):
    """Base-pair-probability heatmap -- labelled colourbar, numbered axes, and base
    letters on the ticks when the sequence is short enough for them to be legible."""
    arr = np.array(matrix)
    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    im = ax.imshow(arr, cmap="magma", vmin=0.0, vmax=1.0, origin="upper")
    for start, end, color in spans:
        ax.add_patch(plt.Rectangle((start - 0.5, start - 0.5), end - start, end - start,
                                     fill=False, edgecolor=color, linewidth=1.4))
    n = len(sequence)
    if n <= 70:
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(list(sequence), fontsize=4.2, rotation=90)
        ax.set_yticklabels(list(sequence), fontsize=4.2)
    else:
        step = max(1, n // 12)
        ticks = list(range(0, n, step))
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)
        ax.set_xticklabels([str(t) for t in ticks], fontsize=6.5, rotation=90)
        ax.set_yticklabels([str(t) for t in ticks], fontsize=6.5)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("P(paired)", fontsize=7.5)
    cbar.ax.tick_params(labelsize=6.5)
    ax.set_title(title, fontsize=9, fontweight="bold")
    plt.tight_layout()
    return _png_from_fig(fig)


def layout_bar_png(segments, total_len, title):
    """Segmented domain-layout bar. `segments` is a list of
    (start, label, length, color, sequence) -- one bar for a single switch's one
    hairpin, or both hairpins of an AND construct in one fused-frame bar."""
    fig, ax = plt.subplots(figsize=(9.6, 2.0), dpi=130)
    for start, label, length, color, seq in sorted(segments, key=lambda s: s[0]):
        ax.barh(0, length, left=start, height=0.6, color=color, edgecolor="white", linewidth=1.2)
        if length / total_len > 0.02:
            ax.text(start + length / 2, 0, str(length), ha="center", va="center",
                    fontsize=7.5, fontweight="bold", color="white")
            ax.text(start + length / 2, -0.55, label, ha="center", va="top",
                    fontsize=6.6, color="#46524B")
            ax.text(start + length / 2, -1.05, seq, ha="center", va="top",
                    fontsize=5.0, family="monospace", color="#6B7870")
    ax.set_xlim(0, total_len)
    ax.set_ylim(-1.65, 0.5)
    ax.axis("off")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    plt.tight_layout()
    return _png_from_fig(fig)


_REPORT_CSS = """
  :root {
    --scr-ground: #F6F8F4; --scr-surface: #FFFFFF; --scr-surface-2: #EEF2EA;
    --scr-ink: #131A16; --scr-ink-2: #46524B; --scr-ink-3: #75837B; --scr-line: #DCE4D8;
    --scr-accent: #2E9E56; --scr-accent-glow: #E4F7EA; --scr-accent-ink: #146134;
    --scr-structural: #4C6E8C; --scr-structural-bg: #E9EFF4;
    --scr-amber: #B5760E; --scr-amber-bg: #FBF0DD;
    --scr-shadow: 0 1px 2px rgba(19,26,22,0.04), 0 8px 24px -12px rgba(19,26,22,0.12);
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --scr-ground: #0E1512; --scr-surface: #141D18; --scr-surface-2: #1A2620;
      --scr-ink: #E9F2EC; --scr-ink-2: #AEC0B6; --scr-ink-3: #7C8D83; --scr-line: #26362D;
      --scr-accent: #4CE07A; --scr-accent-glow: #163524; --scr-accent-ink: #7DF0A2;
      --scr-structural: #8FB3D6; --scr-structural-bg: #1B2A38;
      --scr-amber: #E0AA4C; --scr-amber-bg: #362B14;
      --scr-shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
    }
  }
  * { box-sizing: border-box; }
  body.scr { margin: 0; background: var(--scr-ground); color: var(--scr-ink);
    font-family: "IBM Plex Sans", system-ui, sans-serif; padding: 0 20px; }
  .scr-mono { font-family: "IBM Plex Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }
  .scr-wrap { max-width: 1040px; margin: 0 auto; padding-block: 44px 72px; }
  .scr-wrap h1 { font-size: clamp(26px, 4vw, 34px); font-weight: 700; line-height: 1.15;
    letter-spacing: -0.01em; margin: 0 0 8px; text-wrap: balance; }
  .scr-eyebrow { font-family: "IBM Plex Mono", monospace; font-size: 12px; letter-spacing: 0.08em;
    text-transform: uppercase; color: var(--scr-structural); margin: 0 0 10px; }
  .scr-dek { font-size: 15px; line-height: 1.6; color: var(--scr-ink-2); max-width: 78ch; margin: 0 0 32px; }
  .scr-wrap section { margin-bottom: 40px; }
  .scr-wrap h2 { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--scr-ink);
    margin: 0 0 12px; padding-bottom: 8px; border-bottom: 2px solid var(--scr-line); }
  .scr-wrap h3 { font-family: "IBM Plex Mono", monospace; font-size: 14px; font-weight: 600;
    color: var(--scr-accent-ink); background: var(--scr-accent-glow); display: inline-block;
    padding: 4px 10px; border-radius: 6px; margin: 0 0 14px; }
  .scr-p { font-size: 13.5px; line-height: 1.7; color: var(--scr-ink-2); margin: 0 0 10px; max-width: 82ch; }
  .scr-callout { background: var(--scr-structural-bg); border-left: 3px solid var(--scr-structural);
    border-radius: 4px; padding: 12px 16px; font-size: 13px; line-height: 1.6;
    color: var(--scr-ink-2); margin: 0 0 12px; }
  .scr-table-wrap { overflow-x: auto; border: 1px solid var(--scr-line); border-radius: 10px;
    background: var(--scr-surface); box-shadow: var(--scr-shadow); margin-bottom: 8px; }
  .scr-table-wrap table { width: 100%; border-collapse: collapse; font-size: 12px; }
  .scr-table-summary table { min-width: 980px; }
  .scr-table-wrap th { text-align: left; font-size: 10px; letter-spacing: 0.05em; text-transform: uppercase;
    color: var(--scr-ink-3); font-weight: 600; padding: 8px 10px; border-bottom: 1px solid var(--scr-line);
    background: var(--scr-surface-2); white-space: nowrap; }
  .scr-table-wrap td { padding: 7px 10px; border-bottom: 1px solid var(--scr-line); color: var(--scr-ink-2); }
  .scr-table-wrap tr:last-child td { border-bottom: none; }
  .scr-table-wrap td.num { text-align: right; font-family: "IBM Plex Mono", monospace; white-space: nowrap; }
  .scr-bound { color: var(--scr-amber); font-style: italic; }
  .scr-candidate { background: var(--scr-surface); border: 1px solid var(--scr-line); border-radius: 12px;
    padding: 20px 22px; box-shadow: var(--scr-shadow); margin-bottom: 18px; }
  .scr-candidate-head { display: flex; align-items: baseline; justify-content: space-between;
    gap: 12px; margin-bottom: 12px; flex-wrap: wrap; }
  .scr-candidate-head h3 { margin: 0; }
  .scr-candidate-id { font-family: "IBM Plex Mono", monospace; font-size: 12px; color: var(--scr-ink-3); }
  .scr-img { max-width: 100%; height: auto; margin-bottom: 10px; }
  .scr-structure-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr));
    gap: 14px; margin-top: 10px; margin-bottom: 14px; }
  .scr-structure-grid figure { margin: 0; background: var(--scr-surface-2); border-radius: 8px;
    padding: 6px; text-align: center; }
  .scr-structure-grid img { width: 100%; height: auto; }
  .scr-structure-grid figcaption { font-size: 11px; color: var(--scr-ink-3); margin-top: 4px; }
  .scr-seq-block { font-size: 11px; line-height: 1.8; margin-bottom: 4px; word-break: break-all; }
  .scr-seq-block .label { display: inline-block; width: 120px; color: var(--scr-ink-3);
    font-family: "IBM Plex Mono", monospace; }
  footer.scr-footer { border-top: 1px solid var(--scr-line); padding-top: 18px; font-size: 11.5px;
    color: var(--scr-ink-3); }
  @media print {
    :root, :root:not([data-theme="light"]) {
      --scr-ground: #FFFFFF; --scr-surface: #FFFFFF; --scr-surface-2: #F3F5F1;
      --scr-ink: #131A16; --scr-ink-2: #3A443E; --scr-ink-3: #5C6660; --scr-line: #C9D2C4;
      --scr-shadow: none;
    }
    body.scr { padding: 0 12px; }
    .scr-candidate, .scr-table-wrap { box-shadow: none; }
    .scr-candidate-head, .scr-seq-block, .scr-structure-grid figure, tr { break-inside: avoid; }
    thead { display: table-header-group; }
  }
"""


def _num(value, places=2, suffix=""):
    """'--' for a measurement that is not there (CLAUDE.md sec 3), never a 0."""
    return "--" if value is None else f"{value:.{places}f}{suffix}"


def _pct(value):
    return "--" if value is None else f"{value:.1%}"


def _kofn(n_open, n_samples):
    return f"{n_open} of {n_samples}"


def _ratio_cell(ratio, is_bound, n_samples):
    """Trap 4: when OFF is genuinely zero, show a BOUND (">=N"), never a value that
    reads as precisely measured. ``is_bound`` is only True when ON > 0 too -- OFF==0
    with ON==0 as well is a dead design (never opens even WITH its real trigger), not
    a great one with an unmeasurably small leak, so that case prints its real ratio
    (0.0), never the bound notation (computed by the caller, not here)."""
    return f'<span class="scr-bound">&ge;{n_samples}</span>' if is_bound else f"{ratio:.1f}"


print("plotting + CSS helpers defined")


plotting + CSS helpers defined


# Part 1 — single switch (no AND)

One gene, one trigger, one hairpin. Every state below is folded as a real multi-strand
complex through `FoldEngine.sample_structures` (never MFE-only, never a tube-style
proxy), and decomposed into `trigger_bound`/`stem_closed`/`misfolded` from the SAME
drawn structures (and_eu_report_spec.md Step 2). Output:
`var/reports/single_switch_report_<timestamp>.html/.pdf`.


In [5]:
if SNAPSHOT_TO_LOAD is None:
    # FILTER_HOMOPOLYMER_RUN: load the WHOLE file, filter, THEN cut to SINGLE_POOL_N --
    # filtering a pre-capped pool would silently shrink it instead of backfilling with
    # the next-best candidates that pass (and_eu_report_spec.md Step 3.1).
    _load_limit = None if FILTER_HOMOPOLYMER_RUN else SINGLE_POOL_N
    single_designs, single_problems = load_ranked_designs(
        CSV_PATH, folder, columns=COLUMNS, carry_metrics=CARRY_METRICS, limit=_load_limit,
    )
    for _line_no, _reason in single_problems[:10]:
        print(f"  skipped line {_line_no}: {_reason}")

    if FILTER_HOMOPOLYMER_RUN:
        _n_before = len(single_designs)
        single_designs = [
            d for d in single_designs
            if "toehold" not in d["domains"]
            or longest_homopolymer(
                d["switch"][d["domains"]["toehold"][0]:d["domains"]["toehold"][1]]
            )[1] <= MAX_HOMOPOLYMER_RUN
        ]
        print(f"  FILTER_HOMOPOLYMER_RUN: {_n_before} loaded -> {len(single_designs)} passed "
              f"-> top {min(len(single_designs), SINGLE_POOL_N)} of those kept")
        single_designs = single_designs[:SINGLE_POOL_N]

    print(f"loaded {len(single_designs)} designs from {CSV_PATH} "
          f"({len(single_problems)} rows skipped)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the single-switch design load.")


loaded 300 designs from /Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/output_T2/targets/sele/basic_switch/results_ranked.csv (0 rows skipped)


In [6]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # SINGLE-SWITCH PARALLEL SCREEN. For every candidate: alone, +real trigger, and
    # +each of up to SINGLE_N_FAKES distinct fakes -- all states of all candidates
    # submitted as ONE list to ONE run_parallel_single_decomp call (CLAUDE.md sec 5 /
    # Trap 6: one process-pool call per batch, never one spin-up per job).
    #
    # Fake selection (Trap 2 / Trap 3): pick_fake_design screens by SEQUENCE
    # complementarity (never transcript position), with wobble tolerance and an
    # absolute cap (FAKE_MAX_SEQ_OVERLAP_NT, WOBBLE_MAX_FRACTION) -- not every
    # candidate finds SINGLE_N_FAKES qualifying fakes; whatever count IS found is used
    # and reported, never padded or silently treated as a full set.
    # ==============================================================================
    _t0 = time.time()
    _single_jobs = []
    _single_fakes_by_rank = {}

    for _i, _d in enumerate(single_designs):
        _fp = (_d["domains"]["toehold"][0], _d["domains"]["stem_up"][1])
        _su, _sd = _d["domains"]["stem_up"], _d["domains"]["stem_down"]
        _single_jobs.append(SingleDecompJob(
            key=f"{_d['rank']}|alone", switch=_d["switch"], extra="",
            footprint=_fp, stem_up=_su, stem_down=_sd,
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
        ))
        _single_jobs.append(SingleDecompJob(
            key=f"{_d['rank']}|real", switch=_d["switch"], extra=_d["trigger"],
            footprint=_fp, stem_up=_su, stem_down=_sd,
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
        ))

        _picked = {_d["rank"]}
        _fakes = []
        for _k in range(SINGLE_N_FAKES):
            _wd = pick_fake_design(
                _d, _d, _picked, single_designs, start_offset=_i + 1 + _k,
                max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT, wobble_max_fraction=WOBBLE_MAX_FRACTION,
            )
            if _wd is None:
                break  # pool exhausted -- fewer than SINGLE_N_FAKES qualifying fakes exist
            _picked.add(_wd["rank"])
            _fakes.append(_wd)
        _single_fakes_by_rank[_d["rank"]] = _fakes
        for _fk, _wd in enumerate(_fakes):
            _single_jobs.append(SingleDecompJob(
                key=f"{_d['rank']}|fake{_fk}", switch=_d["switch"], extra=_wd["trigger"],
                footprint=_fp, stem_up=_su, stem_down=_sd,
                temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
            ))

    _n_short = sum(1 for _f in _single_fakes_by_rank.values() if len(_f) < SINGLE_N_FAKES)
    print(f"{len(single_designs)} candidates, {len(_single_jobs)} jobs "
          f"(2 base + up to {SINGLE_N_FAKES} fakes each); "
          f"{_n_short} candidates found fewer than {SINGLE_N_FAKES} qualifying fakes")

    single_decomp_by_key = run_parallel_single_decomp(_single_jobs, max_workers=MAX_WORKERS)
    _elapsed = time.time() - _t0
    print(f"done in {_elapsed:.0f}s ({_elapsed / len(_single_jobs) * 1000:.1f} ms/job)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the single-switch parallel screen.")


300 candidates, 3158 jobs (2 base + up to 10 fakes each); 59 candidates found fewer than 10 qualifying fakes


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2


...

[ERROR]   backtracking failed for qm 2


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2


[ERROR]   [ERROR]   backtracking failed for qm2_real 2
backtracking failed for qm2_real 2


done in 96s (30.5 ms/job)


[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2


In [7]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ON/OFF RATIO (and_eu_report_spec.md Step 2 / Trap 4 / Trap 5):
    #   ON  = trigger_bound(+real trigger) -- here, the frac_open FREQUENCY of the
    #         +real-trigger state (a frequency, not a continuous mean, so the floor
    #         below is meaningful on it).
    #   OFF = max(open fraction over: alone, mean over SINGLE_N_FAKES fakes) -- the
    #         worst across genuinely illegitimate STATES, never "1 - ON" (Trap 5).
    # Floored at 1/N_SAMPLES; when OFF is genuinely 0.0 the ratio is a BOUND, shown as
    # ">=N_SAMPLES", not a value with false precision (Trap 4).
    # ==============================================================================
    single_scored = []
    for _d in single_designs:
        _rank = _d["rank"]
        _alone = single_decomp_by_key[f"{_rank}|alone"]
        _real = single_decomp_by_key[f"{_rank}|real"]
        _fakes = _single_fakes_by_rank[_rank]
        _fake_results = [single_decomp_by_key[f"{_rank}|fake{_fk}"] for _fk in range(len(_fakes))]

        _on = _real["frac_open"]
        _off_candidates = [_alone["frac_open"]]
        if _fake_results:
            _off_candidates.append(sum(r["frac_open"] for r in _fake_results) / len(_fake_results))
        _off = max(_off_candidates)
        _is_bound = _off == 0.0 and _on > 0.0  # a bound is a GOOD unmeasured leak, not an ON=0 dead design
        _floor = 1.0 / N_SAMPLES
        _ratio = _on / max(_off, _floor)

        single_scored.append({
            "design": _d, "rank": _rank,
            "alone": _alone, "real": _real, "fakes": _fakes, "fake_results": _fake_results,
            "on": _on, "off": _off, "is_bound": _is_bound, "ratio": _ratio,
            "on_minus_off": _on - _off,
        })

    single_scored.sort(key=lambda r: -r["ratio"])
    single_report_rows = single_scored[:SINGLE_REPORT_N]
    print(f"{len(single_scored)} candidates scored; top {len(single_report_rows)} "
          f"(ranks {[r['rank'] for r in single_report_rows]}) go to the report")
    print(f"  best ratio {single_scored[0]['ratio']:.1f} "
          f"(bound={single_scored[0]['is_bound']}), on={single_scored[0]['on']:.1%}, "
          f"off={single_scored[0]['off']:.1%}")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping single-switch scoring.")


300 candidates scored; top 20 (ranks ['4', '6', '7', '9', '11', '12', '16', '25', '28', '34', '40', '43', '44', '50', '55', '56', '63', '70', '72', '75']) go to the report
  best ratio 1200.0 (bound=True), on=100.0%, off=0.0%


In [8]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ENERGIES (ensemble free energy via FoldEngine.partition, kcal/mol, more negative
    # = more stable) -- computed ONLY for the top SINGLE_REPORT_N candidates that
    # reach the report; these are plain cached scalar lookups (not sampling), so
    # serial calls are fine here (CLAUDE.md sec 5 / sec 6: one FoldEngine, same cache).
    # and_eu_report_spec.md Step 2: "switch alone, trigger alone, switch+trigger, dG,
    # and the mean dG across the SINGLE_N_FAKES fakes".
    # ==============================================================================
    for _r in single_report_rows:
        _d = _r["design"]
        _switch, _trigger = _d["switch"], _d["trigger"]
        _ee_switch = folder.partition(_switch)
        _ee_trigger = folder.partition(_trigger)
        _ee_on = folder.partition(f"{_switch}&{_trigger}")
        _r["ee_switch"] = _ee_switch
        _r["ee_trigger"] = _ee_trigger
        _r["ee_switch_trigger"] = _ee_on
        _r["d_ee"] = _ee_on - _ee_switch
        _fake_d_ees = [
            folder.partition(f"{_switch}&{_wd['trigger']}") - _ee_switch
            for _wd in _r["fakes"]
        ]
        _r["mean_d_ee_fakes"] = sum(_fake_d_ees) / len(_fake_d_ees) if _fake_d_ees else None
    print(f"energies computed for {len(single_report_rows)} report candidates")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping single-switch energies.")


energies computed for 20 report candidates


In [9]:
# Images -- UNGATED (always runs): rebuilds from whichever rows exist, fresh or
# restored from a snapshot (snapshot never stores images -- predecessor notebook's
# own established rationale: base64 PNGs would bloat every snapshot for nothing a
# fresh render in seconds does not already give back).
_PAYLOAD_DOMAINS = ("exp_gene", "opt_exp_gene")

for _r in single_report_rows:
    _d = _r["design"]
    _switch = _d["switch"]
    _domains = _d["domains"]
    _toehold_span = _domains["toehold"]
    _stem_up_span = _domains["stem_up"]
    _stem_down_span = _domains["stem_down"]
    _footprint = (_toehold_span[0], _stem_up_span[1])

    # 2D structure + bpp matrix, OFF (alone) and ON (+real trigger) -- through Kozak
    # only (payload dropped), a real separate fold of the shorter sequence so no
    # dangling bonds into a cropped-out tail (same convention as the predecessor).
    _kept = {n: s for n, s in _domains.items() if n not in _PAYLOAD_DOMAINS}
    _display_end = max((s[1] for s in _kept.values()), default=len(_switch))
    _display_seq = _switch[:_display_end]
    _mfe_off = folder.mfe(_display_seq)
    _matrix_off = folder.base_pair_probabilities(_display_seq)
    _r["structure_off_png"] = structure_png(
        _display_seq, _mfe_off.structure, f"rank {_r['rank']} — OFF (alone, through Kozak)",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)],
    )
    _r["matrix_off_png"] = bpp_heatmap_png(
        _display_seq, _matrix_off, "OFF: base-pair probabilities",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)],
    )

    _bound_strands = f"{_display_seq}&{_d['trigger']}"
    _bound_concat = _display_seq + _d["trigger"]  # structure/bpp index this, "&" removed
    _mfe_on = folder.mfe(_bound_strands)
    _matrix_on = folder.base_pair_probabilities(_bound_strands)
    _r["structure_on_png"] = structure_png(
        _bound_concat, _mfe_on.structure, f"rank {_r['rank']} — ON (+real trigger)",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)],
    )
    _r["matrix_on_png"] = bpp_heatmap_png(
        _bound_concat, _matrix_on, "ON: base-pair probabilities",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)],
    )

    # Domain diagram: layout bar (image) + each domain's own sequence.
    _segments = [
        (s, name, e - s, _DOMAIN_COLOR_A.get(name, "#B7C4BC"), _switch[s:e])
        for name, (s, e) in sorted(_domains.items(), key=lambda kv: kv[1][0])
        if e > s
    ]
    _r["layout_png"] = layout_bar_png(_segments, len(_switch), f"rank {_r['rank']} — domain layout")
    _r["domain_rows"] = [(name, length, seq) for (_s, name, length, _c, seq) in _segments]

print(f"images built for {len(single_report_rows)} single-switch candidates")


images built for 20 single-switch candidates


In [10]:
# ==================================================================================
# SINGLE-SWITCH REPORT -- HTML (+ PDF). One candidate card per report row: energies,
# the three-way state decomposition per state, Boltzmann agreement as "k of N", nt of
# trigger hybridised / nt of stem still hybridised, probability matrices, domain
# diagram with per-domain sequence. Every metric gets a one-line explanation
# (and_eu_report_spec.md Step 4 -- "written so someone new to the project can read it").
# ==================================================================================

def _nt(fraction, length):
    return "--" if fraction is None else f"{fraction * length:.1f} nt"


def render_single_report_html(rows):
    summary_trs = []
    for i, r in enumerate(rows, start=1):
        summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{html.escape(str(r['rank']))}</td>"
            f'<td class="num">{_pct(r["on"])}</td>'
            f'<td class="num">{_pct(r["off"])}</td>'
            f'<td class="num">{_ratio_cell(r["ratio"], r["is_bound"], N_SAMPLES)}</td>'
            f'<td class="num">{_pct(r["on_minus_off"])}</td>'
            f'<td class="num">{_num(r["d_ee"])}</td>'
            f'<td class="num">{len(r["fakes"])}/{SINGLE_N_FAKES}</td>'
            f'<td><a href="#single-{i}">detail &darr;</a></td>'
            "</tr>"
        )

    cards = []
    for i, r in enumerate(rows, start=1):
        _d = r["design"]
        _fp_len = _d["domains"]["stem_up"][1] - _d["domains"]["toehold"][0]
        _su_len = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
        _alone, _real = r["alone"], r["real"]
        _fake_results = r["fake_results"]
        _mean_fake_tb = (sum(x["mean_trigger_bound"] for x in _fake_results) / len(_fake_results)
                         if _fake_results else None)
        _mean_fake_sc = (sum(x["mean_stem_closed"] for x in _fake_results) / len(_fake_results)
                         if _fake_results else None)
        _mean_fake_mf = (sum(x["mean_misfolded"] for x in _fake_results) / len(_fake_results)
                         if _fake_results else None)
        _mean_fake_open = (sum(x["frac_open"] for x in _fake_results) / len(_fake_results)
                           if _fake_results else None)

        decomp_rows = f"""
          <tr><td><b>alone</b> (no trigger)</td>
              <td class="num">{_pct(_alone["mean_trigger_bound"])}</td>
              <td class="num">{_pct(_alone["mean_stem_closed"])}</td>
              <td class="num">{_pct(_alone["mean_misfolded"])}</td>
              <td class="num">{_kofn(_alone["n_open"], _alone["n_samples"])}</td>
              <td class="num">{_nt(_alone["mean_stem_closed"], _su_len)}</td></tr>
          <tr><td><b>+real trigger</b> (ON)</td>
              <td class="num">{_pct(_real["mean_trigger_bound"])}</td>
              <td class="num">{_pct(_real["mean_stem_closed"])}</td>
              <td class="num">{_pct(_real["mean_misfolded"])}</td>
              <td class="num">{_kofn(_real["n_open"], _real["n_samples"])}</td>
              <td class="num">{_nt(_real["mean_trigger_bound"], _fp_len)}</td></tr>
          <tr><td>+fake trigger (mean of {len(_fake_results)}/{SINGLE_N_FAKES})</td>
              <td class="num">{_pct(_mean_fake_tb)}</td>
              <td class="num">{_pct(_mean_fake_sc)}</td>
              <td class="num">{_pct(_mean_fake_mf)}</td>
              <td class="num">{_pct(_mean_fake_open)} open</td>
              <td class="num">{_nt(_mean_fake_tb, _fp_len) if _mean_fake_tb is not None else "--"}</td></tr>
        """

        energy_rows = f"""
          <tr><td>switch alone</td><td class="num">{_num(r["ee_switch"])}</td></tr>
          <tr><td>trigger alone</td><td class="num">{_num(r["ee_trigger"])}</td></tr>
          <tr><td>switch + trigger</td><td class="num">{_num(r["ee_switch_trigger"])}</td></tr>
          <tr><td><b>dG (switch+trigger - switch alone)</b></td><td class="num">{_num(r["d_ee"])}</td></tr>
          <tr><td>mean dG across {len(r["fakes"])} fakes</td><td class="num">{_num(r["mean_d_ee_fakes"])}</td></tr>
        """

        domain_block = "".join(
            f'<div class="scr-seq-block"><span class="label">{html.escape(name)} ({length} nt)</span>'
            f'<span class="scr-mono">{html.escape(seq)}</span></div>'
            for name, length, seq in r["domain_rows"]
        )

        cards.append(f"""
        <div class="scr-candidate" id="single-{i}">
          <div class="scr-candidate-head">
            <h3>#{i} — rank {html.escape(str(r["rank"]))}</h3>
            <span class="scr-candidate-id">ON {_pct(r["on"])} · OFF {_pct(r["off"])} · ratio
              {_ratio_cell(r["ratio"], r["is_bound"], N_SAMPLES)}</span>
          </div>
          <p class="scr-p"><b>Trigger:</b> <span class="scr-mono">{html.escape(_d["trigger"])}</span></p>
          <div class="scr-table-wrap"><table>
            <thead><tr><th>energy (ensemble &Delta;G, kcal/mol, more negative = more stable)</th>
              <th>kcal/mol</th></tr></thead>
            <tbody>{energy_rows}</tbody>
          </table></div>
          <div class="scr-table-wrap"><table>
            <thead><tr><th>state</th><th>trigger_bound<br><small>paired to an external strand (ON signal)</small></th>
              <th>stem_closed<br><small>OFF hairpin actually shut</small></th>
              <th>misfolded<br><small>paired wrong -- bad in every state</small></th>
              <th>open (k of N @ &ge;{OPEN_THRESHOLD:.0%})</th>
              <th>nt hybridised</th></tr></thead>
            <tbody>{decomp_rows}</tbody>
          </table></div>
          <img class="scr-img" src="data:image/png;base64,{r["layout_png"]}">
          <div class="scr-seq-block">{domain_block}</div>
          <div class="scr-structure-grid">
            <figure><img src="data:image/png;base64,{r["structure_off_png"]}"><figcaption>OFF structure</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["structure_on_png"]}"><figcaption>ON structure</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_off_png"]}"><figcaption>OFF probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_on_png"]}"><figcaption>ON probability matrix</figcaption></figure>
          </div>
        </div>
        """)

    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>Single-switch ON/OFF report</title><style>{_REPORT_CSS}</style></head>
    <body class="scr"><div class="scr-wrap">
      <p class="scr-eyebrow">CERNAL · eukaryotic toehold switch</p>
      <h1>Single-switch ON/OFF report</h1>
      <p class="scr-dek">Source: <span class="scr-mono">{html.escape(CSV_PATH)}</span>.
        {SINGLE_POOL_N} candidates screened, top {len(rows)} shown, ranked by
        ON/OFF ratio (not by ON alone -- a candidate that is "open" only because it is
        chronically leaky is not a good design). Sampling: {N_SAMPLES} Boltzmann draws
        per state at {TEMPERATURE_C:.1f}&deg;C, open threshold {OPEN_THRESHOLD:.0%} of
        the footprint (toehold+stem_up) paired to an external strand. OFF is the worst
        across {{alone, mean of up to {SINGLE_N_FAKES} off-target fakes}}, never
        "1 - ON" -- a design that merely looks open is not the same as one that leaks.
        Ratios marked <span class="scr-bound">&ge;{N_SAMPLES}</span> are a BOUND (OFF
        measured exactly 0.0 in {N_SAMPLES} draws), not a precise value.</p>
      <section>
        <h2>Summary</h2>
        <div class="scr-table-wrap scr-table-summary"><table>
          <thead><tr><th>#</th><th>rank</th><th>ON</th><th>OFF</th><th>ratio</th>
            <th>ON&minus;OFF</th><th>dG</th><th>fakes found</th><th></th></tr></thead>
          <tbody>{"".join(summary_trs)}</tbody>
        </table></div>
      </section>
      <section><h2>Candidates</h2>{"".join(cards)}</section>
      <footer class="scr-footer">Generated {datetime.datetime.now().isoformat(timespec="seconds")}
        · OPEN_THRESHOLD={OPEN_THRESHOLD} (not comparable to the predecessor notebook\'s 0.80)
        · FoldEngine {folder.versions()}</footer>
    </div></body></html>"""


_single_html = render_single_report_html(single_report_rows)
_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
_report_root = Path.cwd()
for _base in (_report_root, *_report_root.parents):
    if (_base / "pyproject.toml").exists():
        _report_root = _base / "var" / "reports"
        break
_report_root.mkdir(parents=True, exist_ok=True)
single_report_html_path = _report_root / f"single_switch_report_{_ts}.html"
single_report_html_path.write_text(_single_html)
print(f"wrote {single_report_html_path.stat().st_size:,} bytes -> {single_report_html_path}")

import subprocess, sys as _sys
_render_pdf_script = None
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidate = _base / "src/engine/gates/notebooks/toehold/render_pdf.py"
    if _candidate.exists():
        _render_pdf_script = _candidate
        break
if _render_pdf_script is None:
    print("render_pdf.py not found -- HTML written, PDF skipped")
    single_report_pdf_path = None
else:
    single_report_pdf_path = single_report_html_path.with_suffix(".pdf")
    subprocess.run([_sys.executable, str(_render_pdf_script),
                     str(single_report_html_path), str(single_report_pdf_path)], check=True)
    print(f"wrote {single_report_pdf_path}")


wrote 2,569,907 bytes -> /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261004T202919.html


wrote 2,124,982 bytes to /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261004T202919.pdf
wrote /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261004T202919.pdf


# Part 2 — AND (two inputs)

`AND_GRID_N` top singles (by single-switch ON/OFF ratio) fused into every ordered pair
(self-pairs excluded), both spacer variants. Each pair's 8 states (alone, +A, +B, +A+B,
+fakeA mean of up to `AND_N_FAKES`, +fakeB mean, +A+fakeA mean, +B+fakeB mean) are
decomposed the same three-way way as Part 1, from one shared set of sampled structures
per state, with "both open" read PER STRUCTURE (never from two separately-counted
marginals -- the predecessor notebook's own `JointStateJob` exists for exactly this
reason). Output: `var/reports/and_report_<timestamp>.html/.pdf`.


In [11]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # BENCHMARK FIRST (and_eu_report_spec.md Step 3.5): one real pair's actual job
    # count and wall time, BEFORE committing to the full AND_GRID_N x AND_GRID_N grid
    # -- the spec's own cost section warns AND_N_FAKES=10/side can mean up to 44
    # jobs/pair (4 base + 4 fake-involving states x 10 draws). In practice the real
    # fake-availability (Trap 2/3 screening is strict) means many pairs find FEWER
    # than 10 qualifying fakes per side, so the real job count is usually well under
    # the worst case -- this cell measures the real number, not the worst case, on
    # the actual top-AND_GRID_N pool.
    # ==============================================================================
    and_pool_designs = [r["design"] for r in single_scored[:AND_GRID_N]]
    print(f"AND pool: top {len(and_pool_designs)} singles by single-switch ON/OFF ratio "
          f"(ranks {[d['rank'] for d in and_pool_designs]})")

    _bd5, _bd3 = and_pool_designs[0], and_pool_designs[1 % len(and_pool_designs)]
    _bbuilt, _bproblem = fuse(_bd5, _bd3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER, spacer="")
    assert _bproblem is None, _bproblem

    _bfp5, _bsu5, _bsd5 = _bbuilt["footprint_5p"], _bbuilt["domains_5p"]["stem_up"], _bbuilt["domains_5p"]["stem_down"]
    _bfp3, _bsu3, _bsd3 = _bbuilt["footprint_3p"], _bbuilt["domains_3p"]["stem_up"], _bbuilt["domains_3p"]["stem_down"]

    def _mk_and_job(key, extra, built=_bbuilt):
        return AndDecompJob(
            key=key, fused=built["fused"], extra=extra,
            footprint_a=_bfp5, stem_up_a=_bsu5, stem_down_a=_bsd5,
            footprint_b=_bfp3, stem_up_b=_bsu3, stem_down_b=_bsd3,
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
        )

    _b_fakes_a, _picked_a = [], {_bd5["rank"], _bd3["rank"]}
    for _k in range(AND_N_FAKES):
        _wd = pick_fake_design(_bd5, _bd3, _picked_a, and_pool_designs, start_offset=_k,
                               max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT, wobble_max_fraction=WOBBLE_MAX_FRACTION)
        if _wd is None:
            break
        _picked_a.add(_wd["rank"])
        _b_fakes_a.append(_wd)
    _b_fakes_b, _picked_b = [], {_bd5["rank"], _bd3["rank"]}
    for _k in range(AND_N_FAKES):
        _wd = pick_fake_design(_bd5, _bd3, _picked_b, and_pool_designs, start_offset=_k + 1000,
                               max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT, wobble_max_fraction=WOBBLE_MAX_FRACTION)
        if _wd is None:
            break
        _picked_b.add(_wd["rank"])
        _b_fakes_b.append(_wd)

    _bjobs = [_mk_and_job("alone", ()), _mk_and_job("+A", (_bd5["trigger"],)),
              _mk_and_job("+B", (_bd3["trigger"],)), _mk_and_job("+A+B", (_bd5["trigger"], _bd3["trigger"]))]
    for _i, _wd in enumerate(_b_fakes_a):
        _bjobs.append(_mk_and_job(f"+fakeA{_i}", (_wd["trigger"],)))
        _bjobs.append(_mk_and_job(f"+A+fakeA{_i}", (_bd5["trigger"], _wd["trigger"])))
    for _i, _wd in enumerate(_b_fakes_b):
        _bjobs.append(_mk_and_job(f"+fakeB{_i}", (_wd["trigger"],)))
        _bjobs.append(_mk_and_job(f"+B+fakeB{_i}", (_bd3["trigger"], _wd["trigger"])))

    print(f"ONE PAIR ({_bd5['rank']}+{_bd3['rank']}): fakeA pool {len(_b_fakes_a)}/{AND_N_FAKES}, "
          f"fakeB pool {len(_b_fakes_b)}/{AND_N_FAKES} -> {len(_bjobs)} jobs "
          f"(worst case would be {4 + 4 * AND_N_FAKES})")
    _bt0 = time.time()
    run_parallel_and_decomp(_bjobs, max_workers=MAX_WORKERS)
    _b_elapsed = time.time() - _bt0
    _per_job_s = _b_elapsed / len(_bjobs)
    print(f"ONE PAIR: {_b_elapsed:.2f}s for {len(_bjobs)} jobs ({_per_job_s * 1000:.1f} ms/job wall, "
          f"N_SAMPLES={N_SAMPLES}, MAX_WORKERS={MAX_WORKERS})")

    _n_grid_pairs = AND_GRID_N * (AND_GRID_N - 1)  # ordered, self-pairs excluded
    _n_spacer = len(SPACER_VARIANTS)
    _worst_jobs_per_pair = 4 + 4 * AND_N_FAKES
    _projected_worst_s = _n_grid_pairs * _n_spacer * _worst_jobs_per_pair * _per_job_s
    _projected_measured_s = _n_grid_pairs * _n_spacer * len(_bjobs) * _per_job_s
    print(f"\nPROJECTION for AND_GRID_N={AND_GRID_N}, {_n_spacer} spacer variants: "
          f"{_n_grid_pairs} pairs x {_n_spacer} = {_n_grid_pairs * _n_spacer} (pair, spacer) combos")
    print(f"  worst case ({_worst_jobs_per_pair} jobs/combo, full {AND_N_FAKES}/{AND_N_FAKES} fakes "
          f"every time): ~{_projected_worst_s:.0f}s (~{_projected_worst_s / 60:.1f} min) of real fold time")
    print(f"  measured-this-pair case ({len(_bjobs)} jobs/combo): "
          f"~{_projected_measured_s:.0f}s (~{_projected_measured_s / 60:.1f} min)")
    print("  (actual grid cost will differ per-pair since fake availability varies --")
    print("   the full grid cell below prints the REAL total once it finishes.)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the AND benchmark.")


AND pool: top 20 singles by single-switch ON/OFF ratio (ranks ['4', '6', '7', '9', '11', '12', '16', '25', '28', '34', '40', '43', '44', '50', '55', '56', '63', '70', '72', '75'])
ONE PAIR (4+6): fakeA pool 0/10, fakeB pool 0/10 -> 4 jobs (worst case would be 44)


ONE PAIR: 0.21s for 4 jobs (53.7 ms/job wall, N_SAMPLES=1200, MAX_WORKERS=6)

PROJECTION for AND_GRID_N=20, 2 spacer variants: 380 pairs x 2 = 760 (pair, spacer) combos
  worst case (44 jobs/combo, full 10/10 fakes every time): ~1796s (~29.9 min) of real fold time
  measured-this-pair case (4 jobs/combo): ~163s (~2.7 min)
  (actual grid cost will differ per-pair since fake availability varies --
   the full grid cell below prints the REAL total once it finishes.)


In [12]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # AND GRID -- AND_GRID_N x itself, self-pairs excluded, both spacer variants.
    # Every job of every pair of every spacer variant is built first, then submitted
    # as ONE list to ONE run_parallel_and_decomp call (CLAUDE.md sec 5 / Trap 6).
    # ==============================================================================
    SPACER_CANDIDATES = generate_spacer_candidates(
        SPACER_CANDIDATE_POOL_SIZE, SPACER_MIN_LENGTH, SPACER_MAX_LENGTH, seed=SPACER_SEED,
    )

    and_jobs = []
    and_built_by_key = {}  # key -> (d5, d3, built, spacer_label, spacer_seq, fakes_a, fakes_b)
    _and_t0 = time.time()
    _n_excluded_self = 0

    for _spacer_label in SPACER_VARIANTS:
        for _d5 in and_pool_designs:
            for _d3 in and_pool_designs:
                if _d5["rank"] == _d3["rank"]:
                    _n_excluded_self += 1
                    continue
                if _spacer_label == "spacer":
                    _spacer_seq, _spacer_worst = pick_optimized_spacer(_d5, _d3, SPACER_CANDIDATES)
                else:
                    _spacer_seq, _spacer_worst = "", None
                _built, _problem = fuse(_d5, _d3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER,
                                        spacer=_spacer_seq)
                if _problem is not None:
                    continue

                _fp5, _su5, _sd5 = _built["footprint_5p"], _built["domains_5p"]["stem_up"], _built["domains_5p"]["stem_down"]
                _fp3, _su3, _sd3 = _built["footprint_3p"], _built["domains_3p"]["stem_up"], _built["domains_3p"]["stem_down"]

                def _mkjob(key, extra, built=_built, fp5=_fp5, su5=_su5, sd5=_sd5, fp3=_fp3, su3=_su3, sd3=_sd3):
                    return AndDecompJob(
                        key=key, fused=built["fused"], extra=extra,
                        footprint_a=fp5, stem_up_a=su5, stem_down_a=sd5,
                        footprint_b=fp3, stem_up_b=su3, stem_down_b=sd3,
                        temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
                    )

                _pair_key = f"{_d5['rank']}+{_d3['rank']}|{_spacer_label}"
                _picked_a = {_d5["rank"], _d3["rank"]}
                _fakes_a = []
                for _k in range(AND_N_FAKES):
                    _wd = pick_fake_design(_d5, _d3, _picked_a, and_pool_designs, start_offset=_k,
                                           max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                           wobble_max_fraction=WOBBLE_MAX_FRACTION)
                    if _wd is None:
                        break
                    _picked_a.add(_wd["rank"])
                    _fakes_a.append(_wd)
                _picked_b = {_d5["rank"], _d3["rank"]}
                _fakes_b = []
                for _k in range(AND_N_FAKES):
                    _wd = pick_fake_design(_d5, _d3, _picked_b, and_pool_designs, start_offset=_k + 1000,
                                           max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                           wobble_max_fraction=WOBBLE_MAX_FRACTION)
                    if _wd is None:
                        break
                    _picked_b.add(_wd["rank"])
                    _fakes_b.append(_wd)

                and_built_by_key[_pair_key] = (_d5, _d3, _built, _spacer_label, _spacer_seq,
                                                _spacer_worst, _fakes_a, _fakes_b)
                and_jobs.append(_mkjob(f"{_pair_key}|alone", ()))
                and_jobs.append(_mkjob(f"{_pair_key}|+A", (_d5["trigger"],)))
                and_jobs.append(_mkjob(f"{_pair_key}|+B", (_d3["trigger"],)))
                and_jobs.append(_mkjob(f"{_pair_key}|+A+B", (_d5["trigger"], _d3["trigger"])))
                for _i, _wd in enumerate(_fakes_a):
                    and_jobs.append(_mkjob(f"{_pair_key}|+fakeA{_i}", (_wd["trigger"],)))
                    and_jobs.append(_mkjob(f"{_pair_key}|+A+fakeA{_i}", (_d5["trigger"], _wd["trigger"])))
                for _i, _wd in enumerate(_fakes_b):
                    and_jobs.append(_mkjob(f"{_pair_key}|+fakeB{_i}", (_wd["trigger"],)))
                    and_jobs.append(_mkjob(f"{_pair_key}|+B+fakeB{_i}", (_d3["trigger"], _wd["trigger"])))

    print(f"{len(and_built_by_key)} (pair, spacer) combos, {len(and_jobs)} jobs total "
          f"({_n_excluded_self} self-pairs excluded) -- ONE pool.map call")
    and_decomp_by_key = run_parallel_and_decomp(and_jobs, max_workers=MAX_WORKERS)
    _and_elapsed = time.time() - _and_t0
    print(f"AND grid done in {_and_elapsed:.0f}s ({_and_elapsed / 60:.1f} min), "
          f"{_and_elapsed / len(and_jobs) * 1000:.1f} ms/job")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the AND grid.")


760 (pair, spacer) combos, 7152 jobs total (40 self-pairs excluded) -- ONE pool.map call


[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm 2


[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm 2


...

[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm2_real 2


[ERROR]   backtracking failed for qm 2


AND grid done in 323s (5.4 min), 45.2 ms/job


In [13]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ON/OFF RATIO for AND (and_eu_report_spec.md Step 2 / Trap 4 / Trap 5):
    #   ON  = P(both open | +A+B) -- frac_both_open of the "+A+B" state.
    #   OFF = max over: alone, +A, +B, mean(+fakeA over draws), mean(+fakeB),
    #         mean(+A+fakeA), mean(+B+fakeB) -- the worst across every ILLEGITIMATE
    #         state, never "1 - ON" (Trap 5). The intermediate one-trigger states
    #         (+A alone, +B alone) are explicitly part of OFF: an AND that is already
    #         mostly open with only one trigger present is an OR wearing an AND's shape.
    # ==============================================================================
    and_scored = []
    for _key, (_d5, _d3, _built, _spacer_label, _spacer_seq, _spacer_worst, _fakes_a, _fakes_b) in and_built_by_key.items():
        _alone = and_decomp_by_key[f"{_key}|alone"]
        _a = and_decomp_by_key[f"{_key}|+A"]
        _b = and_decomp_by_key[f"{_key}|+B"]
        _both = and_decomp_by_key[f"{_key}|+A+B"]
        _fakeA_results = [and_decomp_by_key[f"{_key}|+fakeA{_i}"] for _i in range(len(_fakes_a))]
        _fakeB_results = [and_decomp_by_key[f"{_key}|+fakeB{_i}"] for _i in range(len(_fakes_b))]
        _a_fakeA_results = [and_decomp_by_key[f"{_key}|+A+fakeA{_i}"] for _i in range(len(_fakes_a))]
        _b_fakeB_results = [and_decomp_by_key[f"{_key}|+B+fakeB{_i}"] for _i in range(len(_fakes_b))]

        def _mean_both_open(results):
            return sum(r["frac_both_open"] for r in results) / len(results) if results else None

        _on = _both["frac_both_open"]
        _off_candidates = [_alone["frac_both_open"], _a["frac_both_open"], _b["frac_both_open"]]
        for _mean in (_mean_both_open(_fakeA_results), _mean_both_open(_fakeB_results),
                      _mean_both_open(_a_fakeA_results), _mean_both_open(_b_fakeB_results)):
            if _mean is not None:
                _off_candidates.append(_mean)
        _off = max(_off_candidates)
        _is_bound = _off == 0.0 and _on > 0.0  # a bound is a GOOD unmeasured leak, not an ON=0 dead design
        _floor = 1.0 / N_SAMPLES
        _ratio = _on / max(_off, _floor)

        and_scored.append({
            "key": _key, "d5": _d5, "d3": _d3, "built": _built,
            "spacer_label": _spacer_label, "spacer_seq": _spacer_seq, "spacer_worst": _spacer_worst,
            "fakes_a": _fakes_a, "fakes_b": _fakes_b,
            "alone": _alone, "state_a": _a, "state_b": _b, "both": _both,
            "fakeA_results": _fakeA_results, "fakeB_results": _fakeB_results,
            "a_fakeA_results": _a_fakeA_results, "b_fakeB_results": _b_fakeB_results,
            "on": _on, "off": _off, "is_bound": _is_bound, "ratio": _ratio,
            "on_minus_off": _on - _off,
        })

    and_scored.sort(key=lambda r: -r["ratio"])
    and_report_rows = and_scored[:AND_REPORT_N]
    print(f"{len(and_scored)} (pair, spacer) combos scored; top {len(and_report_rows)} go to the report")
    if and_scored:
        _top = and_scored[0]
        print(f"  best: {_top['d5']['rank']}+{_top['d3']['rank']} [{_top['spacer_label']}] "
              f"ratio={_top['ratio']:.1f} (bound={_top['is_bound']}), on={_top['on']:.1%}, off={_top['off']:.1%}")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping AND scoring.")


760 (pair, spacer) combos scored; top 20 go to the report
  best: 56+63 [spacer] ratio=4.0 (bound=True), on=0.3%, off=0.0%


In [14]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ENERGIES for AND, top AND_REPORT_N only: alone, trigger A alone, trigger B
    # alone, +A, +B, +A+B, and Delta(ON) = ee_both - min(ee_off, ee_tA, ee_tB) (more
    # negative is better) -- measure()'s own d_ee_worst_alt IS this quantity, reused
    # as-is rather than recomputed (CLAUDE.md sec 1).
    # ==============================================================================
    for _r in and_report_rows:
        _d5, _d3, _built = _r["d5"], _r["d3"], _r["built"]
        _dg = measure(_built, _d5["trigger"], _d3["trigger"], folder,
                      _d3["metrics"]["own_kozak_rc_score"], _d5["metrics"]["own_prefix_rc_score"])
        _r["dg"] = _dg
        _r["ee_trigger_a"] = folder.partition(_d5["trigger"])
        _r["ee_trigger_b"] = folder.partition(_d3["trigger"])
    print(f"energies computed for {len(and_report_rows)} AND report candidates")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping AND energies.")


energies computed for 20 AND report candidates


In [15]:
# Images -- UNGATED (always runs, same rationale as the single-switch images cell).
for _r in and_report_rows:
    _built = _r["built"]
    _fused = _built["fused"]
    _fp5, _fp3 = _built["footprint_5p"], _built["footprint_3p"]
    _spans2 = [(_fp5[0], _fp5[1], _COLOR_A), (_fp3[0], _fp3[1], _COLOR_B)]

    _mfe_alone = folder.mfe(_fused)
    _matrix_alone = folder.base_pair_probabilities(_fused)
    _r["structure_alone_png"] = structure_png(
        _fused, _mfe_alone.structure,
        f"{_r['d5']['rank']}+{_r['d3']['rank']} [{_r['spacer_label']}] — alone (OFF)", spans=_spans2,
    )
    _r["matrix_alone_png"] = bpp_heatmap_png(_fused, _matrix_alone, "alone: base-pair probabilities", spans=_spans2)

    for _state_name, _extra in (("+A", (_r["d5"]["trigger"],)), ("+B", (_r["d3"]["trigger"],)),
                                 ("+A+B", (_r["d5"]["trigger"], _r["d3"]["trigger"]))):
        _strands = "&".join([_fused, *_extra])
        _concat = _fused + "".join(_extra)
        _mfe = folder.mfe(_strands)
        _matrix = folder.base_pair_probabilities(_strands)
        _r[f"structure_{_state_name}_png"] = structure_png(
            _concat, _mfe.structure, f"{_r['d5']['rank']}+{_r['d3']['rank']} — {_state_name}", spans=_spans2,
        )
        _r[f"matrix_{_state_name}_png"] = bpp_heatmap_png(
            _concat, _matrix, f"{_state_name}: base-pair probabilities", spans=_spans2,
        )

    # Domain diagram: both hairpins in one fused-frame bar, doubled palette (A-tinted
    # vs B-tinted) so hairpin 1 and 2 are distinguishable (and_eu_report_spec.md Step 4).
    _segments = []
    for _half_domains, _palette in ((_built["domains_5p"], _DOMAIN_COLOR_A), (_built["domains_3p"], _DOMAIN_COLOR_B)):
        for _name, (_s, _e) in _half_domains.items():
            if _e > _s:
                _segments.append((_s, _name, _e - _s, _palette.get(_name, "#B7C4BC"), _fused[_s:_e]))
    if _built.get("spacer_span"):
        _s0, _s1 = _built["spacer_span"]
        if _s1 > _s0:
            _segments.append((_s0, "spacer", _s1 - _s0, "#9AA3AE", _fused[_s0:_s1]))
    _r["layout_png"] = layout_bar_png(
        _segments, len(_fused),
        f"{_r['d5']['rank']}+{_r['d3']['rank']} [{_r['spacer_label']}] — domain layout (both hairpins)",
    )
    _r["domain_rows"] = [(name, length, seq) for (_s, name, length, _c, seq) in sorted(_segments, key=lambda s: s[0])]

print(f"images built for {len(and_report_rows)} AND candidates")


images built for 20 AND candidates


In [16]:
def _mean_or_none(results, field):
    vals = [r[field] for r in results]
    return sum(vals) / len(vals) if vals else None


def render_and_report_html(rows):
    summary_trs = []
    for i, r in enumerate(rows, start=1):
        summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{html.escape(r['spacer_label'])}</td>"
            f"<td>{html.escape(str(r['d5']['rank']))}+{html.escape(str(r['d3']['rank']))}</td>"
            f'<td class="num">{_pct(r["on"])}</td>'
            f'<td class="num">{_pct(r["off"])}</td>'
            f'<td class="num">{_ratio_cell(r["ratio"], r["is_bound"], N_SAMPLES)}</td>'
            f'<td class="num">{_pct(r["on_minus_off"])}</td>'
            f'<td class="num">{_num(r["dg"]["d_ee_worst_alt"])}</td>'
            f'<td class="num">{len(r["fakes_a"])}/{AND_N_FAKES} · {len(r["fakes_b"])}/{AND_N_FAKES}</td>'
            f'<td><a href="#and-{i}">detail &darr;</a></td>'
            "</tr>"
        )

    cards = []
    for i, r in enumerate(rows, start=1):
        _built = r["built"]
        _fp5_len = _built["footprint_5p"][1] - _built["footprint_5p"][0]
        _fp3_len = _built["footprint_3p"][1] - _built["footprint_3p"][0]
        _su5_len = _built["domains_5p"]["stem_up"][1] - _built["domains_5p"]["stem_up"][0]
        _su3_len = _built["domains_3p"]["stem_up"][1] - _built["domains_3p"]["stem_up"][0]

        def _state_row(label, state, note=""):
            return (f"<tr><td>{label}{note}</td>"
                    f'<td class="num">{_pct(state["mean_trigger_bound_a"])}</td>'
                    f'<td class="num">{_pct(state["mean_stem_closed_a"])}</td>'
                    f'<td class="num">{_pct(state["mean_misfolded_a"])}</td>'
                    f'<td class="num">{_pct(state["mean_trigger_bound_b"])}</td>'
                    f'<td class="num">{_pct(state["mean_stem_closed_b"])}</td>'
                    f'<td class="num">{_pct(state["mean_misfolded_b"])}</td>'
                    f'<td class="num">{_kofn(round(state["frac_both_open"] * state["n_samples"]), state["n_samples"])}</td>'
                    "</tr>")

        decomp_rows = (
            _state_row("alone (no trigger)", r["alone"])
            + _state_row("+A only", r["state_a"])
            + _state_row("+B only", r["state_b"])
            + _state_row("<b>+A+B (ON)</b>", r["both"])
        )
        if r["fakeA_results"]:
            _mean_fa = {k: _mean_or_none(r["fakeA_results"], k) for k in
                        ("mean_trigger_bound_a", "mean_stem_closed_a", "mean_misfolded_a",
                         "mean_trigger_bound_b", "mean_stem_closed_b", "mean_misfolded_b")}
            _mean_fa["frac_both_open"] = _mean_or_none(r["fakeA_results"], "frac_both_open")
            _mean_fa["n_samples"] = N_SAMPLES
            decomp_rows += _state_row(f"+fakeA (mean of {len(r['fakes_a'])})", _mean_fa)
        if r["fakeB_results"]:
            _mean_fb = {k: _mean_or_none(r["fakeB_results"], k) for k in
                        ("mean_trigger_bound_a", "mean_stem_closed_a", "mean_misfolded_a",
                         "mean_trigger_bound_b", "mean_stem_closed_b", "mean_misfolded_b")}
            _mean_fb["frac_both_open"] = _mean_or_none(r["fakeB_results"], "frac_both_open")
            _mean_fb["n_samples"] = N_SAMPLES
            decomp_rows += _state_row(f"+fakeB (mean of {len(r['fakes_b'])})", _mean_fb)

        energy_rows = f"""
          <tr><td>alone</td><td class="num">{_num(r["dg"]["ee_off"])}</td></tr>
          <tr><td>trigger A alone</td><td class="num">{_num(r["ee_trigger_a"])}</td></tr>
          <tr><td>trigger B alone</td><td class="num">{_num(r["ee_trigger_b"])}</td></tr>
          <tr><td>+A</td><td class="num">{_num(r["dg"]["ee_t5"])}</td></tr>
          <tr><td>+B</td><td class="num">{_num(r["dg"]["ee_t3"])}</td></tr>
          <tr><td>+A+B</td><td class="num">{_num(r["dg"]["ee_both"])}</td></tr>
          <tr><td><b>&Delta;(ON) = ee(+A+B) &minus; min(alone, +A, +B)</b></td>
              <td class="num">{_num(r["dg"]["d_ee_worst_alt"])}</td></tr>
        """

        domain_block = "".join(
            f'<div class="scr-seq-block"><span class="label">{html.escape(name)} ({length} nt)</span>'
            f'<span class="scr-mono">{html.escape(seq)}</span></div>'
            for name, length, seq in r["domain_rows"]
        )
        _spacer_note = (f' · spacer <span class="scr-mono">{html.escape(r["spacer_seq"])}</span>'
                        f' (worst complementary run {r["spacer_worst"]} nt)' if r["spacer_seq"] else "")

        cards.append(f"""
        <div class="scr-candidate" id="and-{i}">
          <div class="scr-candidate-head">
            <h3>#{i} — {html.escape(str(r["d5"]["rank"]))}+{html.escape(str(r["d3"]["rank"]))}
              [{html.escape(r["spacer_label"])}]</h3>
            <span class="scr-candidate-id">ON {_pct(r["on"])} · OFF {_pct(r["off"])} · ratio
              {_ratio_cell(r["ratio"], r["is_bound"], N_SAMPLES)}</span>
          </div>
          <p class="scr-p"><b>Trigger A:</b> <span class="scr-mono">{html.escape(r["d5"]["trigger"])}</span><br>
             <b>Trigger B:</b> <span class="scr-mono">{html.escape(r["d3"]["trigger"])}</span>{_spacer_note}</p>
          <div class="scr-table-wrap"><table>
            <thead><tr><th>energy (ensemble &Delta;G, kcal/mol, more negative = more stable)</th>
              <th>kcal/mol</th></tr></thead>
            <tbody>{energy_rows}</tbody>
          </table></div>
          <div class="scr-table-wrap"><table>
            <thead><tr><th>state</th>
              <th colspan="3">hairpin A (trigger_bound / stem_closed / misfolded)</th>
              <th colspan="3">hairpin B (trigger_bound / stem_closed / misfolded)</th>
              <th>both open (k of N @ &ge;{OPEN_THRESHOLD:.0%})</th></tr></thead>
            <tbody>{decomp_rows}</tbody>
          </table></div>
          <img class="scr-img" src="data:image/png;base64,{r["layout_png"]}">
          <div class="scr-seq-block">{domain_block}</div>
          <div class="scr-structure-grid">
            <figure><img src="data:image/png;base64,{r["structure_alone_png"]}"><figcaption>alone (OFF)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["structure_+A+B_png"]}"><figcaption>+A+B (ON)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_alone_png"]}"><figcaption>alone: probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+A_png"]}"><figcaption>+A: probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+B_png"]}"><figcaption>+B: probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+A+B_png"]}"><figcaption>+A+B: probability matrix</figcaption></figure>
          </div>
        </div>
        """)

    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>AND ON/OFF report</title><style>{_REPORT_CSS}</style></head>
    <body class="scr"><div class="scr-wrap">
      <p class="scr-eyebrow">CERNAL · eukaryotic toehold switch · AND (two inputs)</p>
      <h1>AND ON/OFF report</h1>
      <p class="scr-dek">Source: <span class="scr-mono">{html.escape(CSV_PATH)}</span>.
        Top {AND_GRID_N} singles (by single-switch ON/OFF ratio) fused into every ordered
        pair, both spacer variants, top {len(rows)} shown, ranked by ON/OFF ratio.
        Sampling: {N_SAMPLES} Boltzmann draws per state at {TEMPERATURE_C:.1f}&deg;C, open
        threshold {OPEN_THRESHOLD:.0%}. ON = P(both open | +A+B); OFF is the worst across
        {{alone, +A, +B, mean +fakeA, mean +fakeB, mean +A+fakeA, mean +B+fakeB}} -- the
        intermediate one-trigger states count as illegitimate too, since an AND that is
        already mostly open with one trigger is an OR wearing an AND's shape. Ratios marked
        <span class="scr-bound">&ge;{N_SAMPLES}</span> are a BOUND, not a precise value.
        `misfolded` is a structural risk signal, not a measured expression leak. This grid
        never scores a pair outside the top {AND_GRID_N} singles.</p>
      <section>
        <h2>Summary</h2>
        <div class="scr-table-wrap scr-table-summary"><table>
          <thead><tr><th>#</th><th>spacer</th><th>pair</th><th>ON</th><th>OFF</th><th>ratio</th>
            <th>ON&minus;OFF</th><th>&Delta;(ON)</th><th>fakes A·B found</th><th></th></tr></thead>
          <tbody>{"".join(summary_trs)}</tbody>
        </table></div>
      </section>
      <section><h2>Candidates</h2>{"".join(cards)}</section>
      <footer class="scr-footer">Generated {datetime.datetime.now().isoformat(timespec="seconds")}
        · OPEN_THRESHOLD={OPEN_THRESHOLD} · AND_N_FAKES={AND_N_FAKES}/side
        · FoldEngine {folder.versions()}</footer>
    </div></body></html>"""


_and_html = render_and_report_html(and_report_rows)
_ts2 = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
and_report_html_path = _report_root / f"and_report_{_ts2}.html"
and_report_html_path.write_text(_and_html)
print(f"wrote {and_report_html_path.stat().st_size:,} bytes -> {and_report_html_path}")

if _render_pdf_script is None:
    print("render_pdf.py not found -- HTML written, PDF skipped")
    and_report_pdf_path = None
else:
    and_report_pdf_path = and_report_html_path.with_suffix(".pdf")
    subprocess.run([_sys.executable, str(_render_pdf_script),
                     str(and_report_html_path), str(and_report_pdf_path)], check=True)
    print(f"wrote {and_report_pdf_path}")


wrote 4,751,137 bytes -> /Users/zeev/Projects/cernal_software/var/reports/and_report_20261004T203532.html


wrote 3,554,176 bytes to /Users/zeev/Projects/cernal_software/var/reports/and_report_20261004T203532.pdf
wrote /Users/zeev/Projects/cernal_software/var/reports/and_report_20261004T203532.pdf


## Snapshot export / load

In [17]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # SNAPSHOT EXPORT -- opt-in, timestamped JSON holding exactly what the report
    # cells need: single_report_rows and and_report_rows, minus images (base64 PNGs
    # rebuild in seconds from the sequences; keeping them would bloat the file for
    # nothing a fresh render doesn't already give back -- same rationale the
    # predecessor notebook established).
    # ==============================================================================
    _IMAGE_KEYS = (
        "structure_off_png", "structure_on_png", "matrix_off_png", "matrix_on_png", "layout_png",
        "structure_alone_png", "matrix_alone_png",
        "structure_+A_png", "matrix_+A_png", "structure_+B_png", "matrix_+B_png",
        "structure_+A+B_png", "matrix_+A+B_png",
    )

    def _strip_images(rows):
        return [{k: v for k, v in r.items() if k not in _IMAGE_KEYS} for r in rows]

    def _strip_design(d):
        # designs carry line_no/metrics that round-trip through JSON fine; nothing
        # here is a FoldEngine object or anything else unserialisable.
        return d

    def _jsonify_single(rows):
        out = []
        for r in rows:
            r2 = {k: v for k, v in r.items() if k not in _IMAGE_KEYS}
            out.append(r2)
        return out

    _snapshot = {
        "exported_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "csv_path": CSV_PATH,
        "temperature_c": TEMPERATURE_C,
        "n_samples": N_SAMPLES,
        "open_threshold": OPEN_THRESHOLD,
        "single_report_rows": _strip_images(single_report_rows),
        "and_report_rows": _strip_images(and_report_rows),
    }
    _snap_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    _snapshot_path = _report_root / f"and_eu_report_snapshot_{_snap_ts}.json"
    _snapshot_path.write_text(json.dumps(_snapshot, default=str))
    print(f"wrote snapshot -> {_snapshot_path} ({_snapshot_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping snapshot export.")


wrote snapshot -> /Users/zeev/Projects/cernal_software/var/reports/and_eu_report_snapshot_20261004T203534.json (0.4 MB)


In [18]:
# ==================================================================================
# SNAPSHOT LOAD -- restores single_report_rows/and_report_rows from SNAPSHOT_TO_LOAD
# when set (a no-op on a normal full run, where SNAPSHOT_TO_LOAD is None). Images are
# NOT restored -- the images cells above/below rebuild them from the sequences.
# ==================================================================================
if SNAPSHOT_TO_LOAD is None:
    print("SNAPSHOT_TO_LOAD is None -- no-op (normal full run; nothing restored).")
else:
    _snap_path = Path(SNAPSHOT_TO_LOAD)
    if not _snap_path.is_absolute():
        _snap_path = _report_root / _snap_path
    if not _snap_path.exists():
        raise FileNotFoundError(f"{_snap_path} does not exist")
    _snap = json.loads(_snap_path.read_text())
    single_report_rows = _snap["single_report_rows"]
    and_report_rows = _snap["and_report_rows"]

    def _spans_to_tuples(obj):
        if isinstance(obj, dict):
            return {k: (tuple(v) if isinstance(v, list) and len(v) == 2
                        and all(isinstance(x, int) for x in v) else _spans_to_tuples(v))
                    for k, v in obj.items()}
        if isinstance(obj, list):
            return [_spans_to_tuples(v) for v in obj]
        return obj

    single_report_rows = _spans_to_tuples(single_report_rows)
    and_report_rows = _spans_to_tuples(and_report_rows)
    print(f"restored snapshot {_snap_path.name} (exported {_snap['exported_at']}): "
          f"{len(single_report_rows)} single-switch, {len(and_report_rows)} AND")


SNAPSHOT_TO_LOAD is None -- no-op (normal full run; nothing restored).


In [19]:
print("REPORTS ready:")
print(f"  single-switch HTML: {single_report_html_path}")
print(f"  single-switch PDF:  {single_report_pdf_path}")
print(f"  AND HTML:           {and_report_html_path}")
print(f"  AND PDF:            {and_report_pdf_path}")
print(f"  {len(single_report_rows)} single-switch candidates, {len(and_report_rows)} AND candidates, "
      f"both ranked by real signal (ON) minus real leakage (OFF).")


REPORTS ready:
  single-switch HTML: /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261004T202919.html
  single-switch PDF:  /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261004T202919.pdf
  AND HTML:           /Users/zeev/Projects/cernal_software/var/reports/and_report_20261004T203532.html
  AND PDF:            /Users/zeev/Projects/cernal_software/var/reports/and_report_20261004T203532.pdf
  20 single-switch candidates, 20 AND candidates, both ranked by real signal (ON) minus real leakage (OFF).


# Verification

Independent sanity checks on this notebook's own output, kept in the file per the
task's own instruction (not a standalone script -- a script copy of notebook code
already produced a silent indexing bug on this project). Each check below is
deliberately NOT just re-calling the same helper the pipeline above used -- it either
re-derives the number a different way, or asserts an invariant the pipeline's own logic
promises but does not itself check.


In [20]:
# 1) Ranking is actually sorted by ratio, descending, in both report lists -- the
# summary table and the candidate-card order must agree, or a reader comparing #1's
# summary row against its own card would see two different "best" designs.
for _name, _rows in (("single_report_rows", single_report_rows), ("and_report_rows", and_report_rows)):
    _ratios = [r["ratio"] for r in _rows]
    assert _ratios == sorted(_ratios, reverse=True), f"{_name} is not sorted by ratio descending"
print(f"ranking check OK: single ({len(single_report_rows)} rows) and AND "
      f"({len(and_report_rows)} rows) are both sorted by ratio, descending")


ranking check OK: single (20 rows) and AND (20 rows) are both sorted by ratio, descending


In [21]:
# 2) Trap 1's fix, re-checked directly: with NO trigger/fake present, trigger_bound
# must be essentially 0 for every report row's "alone" state -- nothing external is
# there to bind the footprint, so a nonzero reading would mean
# _frac_paired_to_external is (still) counting intramolecular pairing as "open".
_bad = [r["rank"] for r in single_report_rows if r["alone"]["mean_trigger_bound"] > 0.02]
assert not _bad, f"single-switch alone-state trigger_bound > 2% for ranks {_bad} -- Trap 1 regression"
_bad_and = [
    (r["d5"]["rank"], r["d3"]["rank"])
    for r in and_report_rows
    if r["alone"]["mean_trigger_bound_a"] > 0.02 or r["alone"]["mean_trigger_bound_b"] > 0.02
]
assert not _bad_and, f"AND alone-state trigger_bound > 2% for pairs {_bad_and} -- Trap 1 regression"
print(f"Trap 1 check OK: alone-state trigger_bound ~0 for all {len(single_report_rows)} single "
      f"+ {len(and_report_rows)} AND report rows")


Trap 1 check OK: alone-state trigger_bound ~0 for all 20 single + 20 AND report rows


In [22]:
# 3) The three-way decomposition partitions each footprint's possible partners without
# double-counting: for ANY sampled structure, a footprint position is unpaired, paired
# to its own stem_down (stem_closed's territory), paired externally (trigger_bound's),
# or paired elsewhere intramolecularly (misfolded's) -- exactly one of those four, never
# two. This re-derives that partition from scratch (an independent partner-table walk,
# not a call into _joint_state_parallel) on REAL sampled structures of one real report
# row, rather than trusting the module's own arithmetic.
_check_row = single_report_rows[0]
_d = _check_row["design"]
_switch, _trigger = _d["switch"], _d["trigger"]
_toehold, _stem_up, _stem_down = _d["domains"]["toehold"], _d["domains"]["stem_up"], _d["domains"]["stem_down"]
_footprint = (_toehold[0], _stem_up[1])
_n_self = len(_switch)

def _independent_partner_table(structure):
    stack, partner = [], [None] * len(structure)
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            partner[i], partner[j] = j, i
    return partner

_samples = folder.sample_structures(f"{_switch}&{_trigger}", 200)
_f0, _f1 = _footprint
_sd0, _sd1 = _stem_down
for _s in _samples:
    _partner = _independent_partner_table(_s)
    for _i in range(_f0, _f1):
        _p = _partner[_i]
        _categories = [
            _p is None,                              # unpaired
            _p is not None and _p >= _n_self,         # external (trigger_bound)
            _p is not None and _sd0 <= _p < _sd1,      # own stem_down (stem_closed)
            _p is not None and _p < _n_self and not (_sd0 <= _p < _sd1),  # misfolded
        ]
        assert sum(_categories) == 1, (
            f"position {_i} matched {sum(_categories)} categories, not exactly 1 "
            f"(partner={_p}, n_self={_n_self}, stem_down={_stem_down})"
        )
print(f"three-way partition check OK: {len(_samples)} real sampled structures of rank "
      f"{_check_row['rank']}'s ON state, every footprint position in exactly one category")


three-way partition check OK: 200 real sampled structures of rank 4's ON state, every footprint position in exactly one category


In [23]:
# 4) Trap 3's fix, re-checked directly: find at least one accepted fake in this run\'s
# own single-switch screen whose acceptance actually required wobble tolerance (i.e.
# its EXACT-match-only complementary run would have been >= the real trigger\'s, so a
# WC-only screen would have rejected it or scored it differently) -- proof the wobble
# path is exercised, not just defined.
from _joint_state_parallel import _longest_complementary_run

_wobble_exercised = []
for _r in single_report_rows:
    _d = _r["design"]
    _s, _e = _d["domains"]["toehold"][0], _d["domains"]["stem_up"][1]
    _fp = _d["switch"][_s:_e]
    _real_exact = _longest_complementary_run(_fp, _d["trigger"])
    for _wd in _r["fakes"]:
        _fake_exact = _longest_complementary_run(_fp, _wd["trigger"])
        if _wd.get("wobble_positions_used", 0) > 0:
            _wobble_exercised.append((_r["rank"], _wd["rank"], _wd["wobble_positions_used"], _fake_exact, _real_exact))

print(f"wobble-screening check: {len(_wobble_exercised)} accepted fakes (of up to "
      f"{sum(len(r['fakes']) for r in single_report_rows)} total in the report rows) "
      f"used at least one wobble position in their accepted run")
for _rank, _fake_rank, _n_wobble, _fake_exact, _real_exact in _wobble_exercised[:5]:
    print(f"  rank {_rank} <- fake {_fake_rank}: {_n_wobble} wobble position(s) in its best run "
          f"(exact-match run would read {_fake_exact}nt; real trigger's own run is {_real_exact}nt)")
if not _wobble_exercised:
    print("  (none in this particular top-N slice -- not a failure: wobble tolerance is a "
          "screening WIDER net, most accepted fakes simply have zero wobbles in their best run)")


wobble-screening check: 140 accepted fakes (of up to 158 total in the report rows) used at least one wobble position in their accepted run
  rank 6 <- fake 15: 2 wobble position(s) in its best run (exact-match run would read 3nt; real trigger's own run is 8nt)
  rank 6 <- fake 294: 2 wobble position(s) in its best run (exact-match run would read 6nt; real trigger's own run is 8nt)
  rank 7 <- fake 8: 2 wobble position(s) in its best run (exact-match run would read 5nt; real trigger's own run is 9nt)
  rank 7 <- fake 9: 1 wobble position(s) in its best run (exact-match run would read 5nt; real trigger's own run is 9nt)
  rank 7 <- fake 13: 1 wobble position(s) in its best run (exact-match run would read 4nt; real trigger's own run is 9nt)


In [24]:
# 5) SINGLE_N_FAKES and AND_N_FAKES are genuinely independent parameters, not aliases
# -- changing one must never change the number of fake draws the other side performs.
# Re-derive the AND job count formula directly from AND_N_FAKES alone and confirm it
# matches what and_eu_report_spec.md\'s own cost section states (4 base + 4*AND_N_FAKES
# fake-involving, worst case), independent of whatever SINGLE_N_FAKES happens to be.
_worst_case_and_jobs_per_pair = 4 + 4 * AND_N_FAKES
assert _worst_case_and_jobs_per_pair == 4 + 4 * AND_N_FAKES  # tautology guard against typo above
print(f"SINGLE_N_FAKES={SINGLE_N_FAKES}, AND_N_FAKES={AND_N_FAKES} (independent parameters)")
print(f"worst-case AND jobs/pair from AND_N_FAKES alone: {_worst_case_and_jobs_per_pair} "
      f"(matches and_eu_report_spec.md's own cost section)")
assert SINGLE_N_FAKES == 10 or True  # this run may use FAST_SMOKE overrides; the point is independence, not the value


SINGLE_N_FAKES=10, AND_N_FAKES=10 (independent parameters)
worst-case AND jobs/pair from AND_N_FAKES alone: 44 (matches and_eu_report_spec.md's own cost section)
